# Fraud Shield - Deep Learning Models

Trains the deep learning branch -- a Feedforward Neural Network (FNN) on
tabular features, and an LSTM over each cardholder's recent transaction
history -- using the model classes in `src/models/deep_learning.py`.

**Important:** this notebook re-creates the *exact same* train/validation
split as `03_baseline_models.ipynb` (same `random_state=42`, `test_size=0.2`,
on the same `train_features.parquet`) so that validation-row indices line
up across notebooks. That alignment is what lets
`05_hybrid_ensemble.ipynb` combine both branches' probabilities per
transaction.

**Inputs:** `data/processed/train_features.parquet`
**Outputs:** trained FNN/LSTM weights + validation predictions in
`data/processed/models/`


In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from src.models.deep_learning import FraudFNN, FraudLSTM
from src.models.evaluate import compute_metrics, print_report

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')  # should report your RTX 4060 if CUDA is set up

MODELS_DIR = Path('../data/processed/models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)


## 1. Load features and reproduce the baseline split


In [ ]:
train_df = pd.read_parquet('../data/processed/train_features.parquet')

id_cols = ['transaction_id', 'event_time', 'cc_num']
target_col = 'is_fraud'
feature_cols = [c for c in train_df.columns if c not in id_cols + [target_col]]

X = train_df[feature_cols]
y = train_df[target_col]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f'Train: {X_train.shape}, fraud rate {y_train.mean()*100:.4f}%')
print(f'Val:   {X_val.shape}, fraud rate {y_val.mean()*100:.4f}%')


## 2. Scale features

Tree models don't need this, but neural nets train far better on
standardized inputs. Fit the scaler on the training split only, then
apply it to validation to avoid leakage.


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

import joblib
joblib.dump(scaler, MODELS_DIR / 'feature_scaler.joblib')


## 3. FNN - data loaders


In [ ]:
batch_size = 512

train_tensor_X = torch.tensor(X_train_scaled, dtype=torch.float32)
train_tensor_y = torch.tensor(y_train.values, dtype=torch.float32)
val_tensor_X = torch.tensor(X_val_scaled, dtype=torch.float32)
val_tensor_y = torch.tensor(y_val.values, dtype=torch.float32)

train_loader = DataLoader(
    TensorDataset(train_tensor_X, train_tensor_y), batch_size=batch_size, shuffle=True
)
val_loader = DataLoader(
    TensorDataset(val_tensor_X, val_tensor_y), batch_size=batch_size, shuffle=False
)


## 4. Train the FNN

Uses `BCELoss` with a `pos_weight`-equivalent manual class weighting
(implemented via a weighted loss per batch) since `FraudFNN` outputs a
sigmoid probability directly rather than a logit.


In [ ]:
fnn = FraudFNN(input_dim=X_train_scaled.shape[1]).to(device)
optimizer = torch.optim.Adam(fnn.parameters(), lr=1e-3)

# Class weight: upweight the rare fraud class in the loss
pos_weight_value = (y_train == 0).sum() / max((y_train == 1).sum(), 1)

def weighted_bce(pred, target, pos_weight):
    weights = torch.where(target == 1, pos_weight, 1.0)
    eps = 1e-7
    pred = torch.clamp(pred, eps, 1 - eps)
    loss = -(target * torch.log(pred) + (1 - target) * torch.log(1 - pred))
    return (loss * weights).mean()

n_epochs = 5
fnn_history = []

for epoch in range(n_epochs):
    fnn.train()
    epoch_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = fnn(xb)
        loss = weighted_bce(preds, yb, pos_weight_value)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * xb.size(0)
    epoch_loss /= len(train_loader.dataset)
    fnn_history.append(epoch_loss)
    print(f'Epoch {epoch+1}/{n_epochs} - train loss: {epoch_loss:.4f}')


## 5. Evaluate the FNN


In [ ]:
fnn.eval()
with torch.no_grad():
    fnn_proba = fnn(val_tensor_X.to(device)).cpu().numpy()
fnn_pred = (fnn_proba >= 0.5).astype(int)

fnn_metrics = compute_metrics(y_val, fnn_pred, fnn_proba)
print(fnn_metrics)
print_report(y_val, fnn_pred)


## 6. Build sequences for the LSTM

The LSTM treats each transaction as the *last* step of a sequence of that
cardholder's `seq_len` most recent transactions (in chronological order,
zero-padded if the cardholder has less history than `seq_len`). Built with
`groupby().shift()` rather than a Python loop, so it scales to the full
~1M-row training set.


In [ ]:
SEQ_LEN = 5

def build_sequences(df, feature_cols, seq_len):
    sorted_df = df.sort_values(['cc_num', 'event_time']).reset_index()
    sorted_df = sorted_df.rename(columns={'index': 'orig_index'})

    lag_arrays = [sorted_df[feature_cols].values]  # lag 0 = current transaction
    for lag in range(1, seq_len):
        shifted = sorted_df.groupby('cc_num')[feature_cols].shift(lag).fillna(0).values
        lag_arrays.append(shifted)

    lag_arrays = lag_arrays[::-1]  # oldest -> newest, current transaction last
    sequences = np.stack(lag_arrays, axis=1)  # (n_rows, seq_len, n_features)

    pos_lookup = pd.Series(np.arange(len(sorted_df)), index=sorted_df['orig_index'])
    return sequences, pos_lookup

sequences, pos_lookup = build_sequences(train_df, feature_cols, SEQ_LEN)
print(f'Sequence tensor shape: {sequences.shape}')


In [ ]:
train_positions = pos_lookup.loc[X_train.index].values
val_positions = pos_lookup.loc[X_val.index].values

X_train_seq_raw = sequences[train_positions]
X_val_seq_raw = sequences[val_positions]

# Sanity check: targets pulled via the same positions should match y_train/y_val exactly
assert np.array_equal(train_df.iloc[train_positions][target_col].values, y_train.values)
assert np.array_equal(train_df.iloc[val_positions][target_col].values, y_val.values)
print('Sequence/label alignment verified.')


## 7. Scale sequences and build the LSTM data loaders

Reuses the same `scaler` fit on the tabular features, applied per
timestep.


In [ ]:
n_train, seq_len, n_features = X_train_seq_raw.shape
n_val = X_val_seq_raw.shape[0]

X_train_seq_scaled = scaler.transform(X_train_seq_raw.reshape(-1, n_features)).reshape(n_train, seq_len, n_features)
X_val_seq_scaled = scaler.transform(X_val_seq_raw.reshape(-1, n_features)).reshape(n_val, seq_len, n_features)

train_seq_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_train_seq_scaled, dtype=torch.float32),
        torch.tensor(y_train.values, dtype=torch.float32),
    ),
    batch_size=batch_size, shuffle=True,
)
val_seq_tensor_X = torch.tensor(X_val_seq_scaled, dtype=torch.float32)


## 8. Train the LSTM


In [ ]:
lstm = FraudLSTM(input_dim=n_features).to(device)
lstm_optimizer = torch.optim.Adam(lstm.parameters(), lr=1e-3)

lstm_history = []

for epoch in range(n_epochs):
    lstm.train()
    epoch_loss = 0.0
    for xb, yb in train_seq_loader:
        xb, yb = xb.to(device), yb.to(device)
        lstm_optimizer.zero_grad()
        preds = lstm(xb)
        loss = weighted_bce(preds, yb, pos_weight_value)
        loss.backward()
        lstm_optimizer.step()
        epoch_loss += loss.item() * xb.size(0)
    epoch_loss /= len(train_seq_loader.dataset)
    lstm_history.append(epoch_loss)
    print(f'Epoch {epoch+1}/{n_epochs} - train loss: {epoch_loss:.4f}')


## 9. Evaluate the LSTM


In [ ]:
lstm.eval()
with torch.no_grad():
    lstm_proba = lstm(val_seq_tensor_X.to(device)).cpu().numpy()
lstm_pred = (lstm_proba >= 0.5).astype(int)

lstm_metrics = compute_metrics(y_val, lstm_pred, lstm_proba)
print(lstm_metrics)
print_report(y_val, lstm_pred)


## 10. Compare FNN vs LSTM (and training curves)


In [ ]:
dl_results = pd.DataFrame({'FNN': fnn_metrics, 'LSTM': lstm_metrics}).T
dl_results = dl_results.sort_values('f1', ascending=False)
dl_results


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(fnn_history, label='FNN', marker='o')
axes[0].plot(lstm_history, label='LSTM', marker='o')
axes[0].set_title('Training loss by epoch')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Weighted BCE loss')
axes[0].legend()

dl_results[['precision', 'recall', 'f1', 'auc_roc']].plot(kind='bar', ax=axes[1])
axes[1].set_title('FNN vs LSTM - validation metrics')
axes[1].legend(loc='lower right')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 11. Compare against the baseline supervised models

Loads `baseline_val_predictions.parquet` from `03_baseline_models.ipynb`.
Because both notebooks split `train_features.parquet` identically
(`random_state=42`, `test_size=0.2`, same feature columns), the row order
here matches `y_val` there -- this is what makes the direct comparison
(and later, the ensemble stacking) valid.


In [ ]:
baseline_val = pd.read_parquet(MODELS_DIR / 'baseline_val_predictions.parquet')

assert len(baseline_val) == len(y_val), 'Row count mismatch -- rerun 03 and 04 against the same train_features.parquet'
assert np.array_equal(baseline_val['is_fraud'].values, y_val.values), 'Row order mismatch -- split parameters must match exactly between notebooks'

combined = pd.DataFrame({
    'Logistic Regression': compute_metrics(y_val, (baseline_val['logreg_proba'] >= 0.5).astype(int), baseline_val['logreg_proba']),
    'Random Forest': compute_metrics(y_val, (baseline_val['rf_proba'] >= 0.5).astype(int), baseline_val['rf_proba']),
    'XGBoost': compute_metrics(y_val, (baseline_val['xgb_proba'] >= 0.5).astype(int), baseline_val['xgb_proba']),
    'FNN': fnn_metrics,
    'LSTM': lstm_metrics,
}).T.sort_values('f1', ascending=False)

combined


## 12. Save deep learning models and validation predictions


In [ ]:
torch.save(fnn.state_dict(), MODELS_DIR / 'fnn.pt')
torch.save(lstm.state_dict(), MODELS_DIR / 'lstm.pt')

dl_val_predictions = pd.DataFrame({
    'transaction_id': train_df.iloc[val_positions]['transaction_id'].values,
    'is_fraud': y_val.values,
    'fnn_proba': fnn_proba,
    'lstm_proba': lstm_proba,
})
dl_val_predictions.to_parquet(MODELS_DIR / 'dl_val_predictions.parquet', index=False)

print('Saved fnn.pt, lstm.pt, and dl_val_predictions.parquet to', MODELS_DIR)


## 13. Next steps

- Move to `05_hybrid_ensemble.ipynb`: stack
  `baseline_val_predictions.parquet` and `dl_val_predictions.parquet`
  probabilities into a meta-learner
- If the LSTM underperforms the FNN, consider a longer `SEQ_LEN`, more
  epochs, or verify that `txn_velocity` (already time-aware) isn't already
  capturing most of the sequential signal the LSTM would otherwise add
- `n_epochs=5` is a fast baseline run -- increase if the loss curve hasn't
  flattened
